# Mission 3 P0 baseline re-run v2 (TF-IDF + OvR Logistic Regression)

Repo-relative version of the earlier local notebook: **no absolute paths in code cells**.
- Repo root: `git rev-parse --show-toplevel` (fallback: walk up to `.git` / `scripts/`).
- Data: `DATA_ROOT` env var, default `data/raw_merged/Training/2.라벨링데이터/TL_서울_구급` (relative to repo root).
- Outputs: `outputs/m3_p0_rerun_v2/` (gitignored).
- Script `scripts/mission3_baseline 최종 수정본.py` is run unmodified (sha256 asserted).

Run from the repo root (this notebook is committed and pushed before execution; cell outputs are the submission log):

```
set PYTHONUNBUFFERED=1
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 notebooks/mission3/mission3_p0_rerun.ipynb --output-dir outputs/m3_p0_rerun_v2 --output mission3_p0_rerun_executed.ipynb
```

Reference: Macro F1 0.5676 (tol ±0.003), N=29,200, 0 load failures.

In [ ]:
# Cell 0: config + provenance
import os, sys, platform, hashlib, subprocess, json, time
from pathlib import Path

os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUNBUFFERED"] = "1"


def find_repo_root(start=None):
    """`git rev-parse --show-toplevel`; fallback: walk up to a folder containing .git or scripts/."""
    start = Path(start) if start else Path.cwd()
    try:
        top = subprocess.run(["git", "rev-parse", "--show-toplevel"], cwd=start, capture_output=True,
                             text=True, encoding="utf-8", errors="replace", check=True).stdout.strip()
        if top:
            return Path(top)
    except (OSError, subprocess.CalledProcessError):
        pass
    for p in (start.resolve(), *start.resolve().parents):
        if (p / ".git").exists() or (p / "scripts").is_dir():
            return p
    raise RuntimeError("repo root not found (run this notebook from inside the repo)")


REPO = find_repo_root()
os.chdir(REPO)   # everything below is relative to the repo root


def rel(p):
    """Repo-relative POSIX string for printing/logging (falls back to str(p) outside the repo)."""
    p = Path(p)
    if not p.is_absolute():
        return p.as_posix()
    try:
        return p.relative_to(REPO).as_posix()
    except ValueError:
        return str(p)


def git(*args):
    return subprocess.run(["git", *args], cwd=REPO, capture_output=True,
                          text=True, encoding="utf-8", errors="replace").stdout.strip()


def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


# Data: DATA_ROOT env var (absolute or repo-relative); default is the repo-relative data/ folder.
DATA_ROOT = os.environ.get("DATA_ROOT", "data/raw_merged/Training/2.라벨링데이터/TL_서울_구급")
LABEL_DIR = Path(DATA_ROOT) if Path(DATA_ROOT).is_absolute() else REPO / DATA_ROOT
SCRIPT = Path("scripts") / "mission3_baseline 최종 수정본.py"
EXPECTED_SCRIPT_SHA = "fee2bef3d52568fd8ecfb31773f0b8987a3dc4a3517508d858fdac9fc7209103"
OUT = Path("outputs") / "m3_p0_rerun_v2"
OUT.mkdir(parents=True, exist_ok=True)
EXPECTED_N = 29200
REF_MACRO_F1 = 0.5676
TOL = 0.003
PREV_PRED_SHA256 = "4884c032d3fa1e9bed772c506c3bdcb0484d202d253b09acaf1cb59f12850b08"   # validation_predictions.csv of the earlier executed run (commit a1ad073)

HEAD = git("rev-parse", "HEAD")
print("repo root         :", REPO.name, "(git toplevel)")
print("git HEAD          :", HEAD)
print("git log -1        :", git("log", "-1", "--oneline"))
print("git branch        :", git("rev-parse", "--abbrev-ref", "HEAD"))
print("HEAD on remote    :", git("branch", "-r", "--contains", HEAD) or "(NOT PUSHED)")
st = git("status", "--porcelain", "--", "scripts/", "notebooks/")
print("git status scripts/ notebooks/:", st if st else "(clean)")
SCRIPT_SHA = sha256(SCRIPT)
print("script            :", rel(SCRIPT))
print("script sha256     :", SCRIPT_SHA, "(matches expected)" if SCRIPT_SHA == EXPECTED_SCRIPT_SHA else "(MISMATCH)")
assert SCRIPT_SHA == EXPECTED_SCRIPT_SHA, "script was modified"
print("python            :", sys.version.replace("\n", " "))
print("executable        :", sys.executable.replace(str(Path.home()), "~"))
print("platform          :", platform.platform(), "|", platform.processor())
print("DATA_ROOT env     :", "(set)" if "DATA_ROOT" in os.environ else "(unset -> default)")
print("LABEL_DIR         :", rel(LABEL_DIR))
n_json = sum(1 for _ in LABEL_DIR.rglob("*.json"))
print("JSON count        :", n_json)
assert n_json == EXPECTED_N, f"expected {EXPECTED_N} JSON, got {n_json}"
print("OUT               :", rel(OUT))


In [ ]:
# Cell 1: dependency check (local: install iterative-stratification only if missing)
try:
    import iterstrat
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "iterative-stratification"], check=True)
    import iterstrat
import sklearn, numpy, pandas, scipy, joblib
print("iterstrat", getattr(iterstrat, "__version__", "(no __version__)"), "|", rel(Path(iterstrat.__file__)).replace(str(Path.home()), "~"))
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scipy", scipy.__version__, "| joblib", joblib.__version__)


In [ ]:
# Cell 2: streaming run() helper (stdout streamed line by line, PYTHONUNBUFFERED, logged to run_log.txt)
LOG = OUT / "run_log.txt"

def run(args):
    args = [str(a) for a in args]
    cmd = [sys.executable, "-u", *args]
    shown = " ".join(f'"{c}"' if " " in c else c for c in ["python", "-u", *args])
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    print("$", shown, flush=True)
    t0 = time.time()
    with open(LOG, "a", encoding="utf-8") as log:
        log.write("$ " + shown + "\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                env=env, text=True, encoding="utf-8", errors="replace",
                                bufsize=1, cwd=str(REPO))
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        rc = proc.wait()
        el = time.time() - t0
        log.write(f"[returncode={rc} elapsed={el:.1f}s]\n")
    print(f"[returncode={rc} elapsed={el:.1f}s]")
    assert rc == 0, f"command failed with returncode {rc}"
    return el


In [ ]:
# Cell 3: train + evaluate P0 baseline (repo-relative args, cwd = repo root)
ELAPSED = run([rel(SCRIPT), "--label_dir", rel(LABEL_DIR), "--output_dir", rel(OUT),
               "--test_size", "0.2", "--seed", "42", "--threshold", "0.5"])
print(f"training wall time: {ELAPSED:.1f}s ({ELAPSED/60:.1f} min)")


In [ ]:
# Cell 4: verdict
import pandas as pd
metrics = json.loads((OUT / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
per_label = pd.read_csv(OUT / "per_label_metrics.csv", encoding="utf-8-sig")
print(per_label.to_string(index=False))

REF_PER_LABEL = {"열상": 0.804, "복통": 0.736, "호흡곤란": 0.631, "어지러움": 0.607, "고열": 0.601,
                 "전신쇠약": 0.523, "구토": 0.486, "두통": 0.409, "오심": 0.312}
f1_col = next((c for c in per_label.columns if "f1" in c.lower()), None)
lab_col = per_label.columns[0]
if f1_col is not None:
    cmp = per_label[[lab_col, f1_col]].copy()
    cmp["ref_f1"] = cmp[lab_col].map(REF_PER_LABEL)
    cmp["diff"] = (cmp[f1_col] - cmp["ref_f1"]).round(4)
    print("\nper-label F1 vs reference:")
    print(cmp.to_string(index=False))

CKPT = OUT / "mission3_tfidf_logreg.joblib"
CKPT_SHA = sha256(CKPT)
print("\nckpt:", rel(CKPT))
print("ckpt sha256:", CKPT_SHA)

PRED = OUT / "validation_predictions.csv"
PRED_SHA = sha256(PRED)
PRED_MATCH = PRED_SHA == PREV_PRED_SHA256
print("validation_predictions.csv sha256:", PRED_SHA)
print("  identical to earlier run       :", PRED_MATCH)

macro = float(metrics["macro_f1"])
diff = macro - REF_MACRO_F1
passed = bool(abs(diff) <= TOL and metrics["total_sample_count"] == EXPECTED_N
              and metrics["load_failure_count"] == 0)
verdict = {"macro_f1": macro, "ref": REF_MACRO_F1, "diff": diff, "tol": TOL, "pass": passed,
           "ckpt": rel(CKPT), "ckpt_sha256": CKPT_SHA, "repo_commit": HEAD,
           "notebook": "notebooks/mission3/mission3_p0_rerun.ipynb",
           "script": rel(SCRIPT), "script_sha256": SCRIPT_SHA, "label_dir": rel(LABEL_DIR),
           "micro_f1": metrics.get("micro_f1"),
           "train_sample_count": metrics.get("train_sample_count"),
           "validation_sample_count": metrics.get("validation_sample_count"),
           "total_sample_count": metrics.get("total_sample_count"),
           "load_failure_count": metrics.get("load_failure_count"),
           "predictions_sha256": PRED_SHA, "prev_predictions_sha256": PREV_PRED_SHA256,
           "predictions_match_prev_run": PRED_MATCH,
           "elapsed_sec": round(ELAPSED, 1)}
(OUT / "verdict.json").write_text(json.dumps(verdict, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(verdict, ensure_ascii=False, indent=2))
print(f"\nVERDICT: {'PASS' if passed else 'FAIL'}  macro_f1={macro:.5f} ref={REF_MACRO_F1} diff={diff:+.5f} tol={TOL}")


In [ ]:
# Cell 5: pip freeze
import re
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True,
                        text=True, encoding="utf-8", errors="replace").stdout
(OUT / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
print("wrote", rel(OUT / "pip_freeze.txt"), f"({len(freeze.splitlines())} packages)")
for line in freeze.splitlines():
    if re.search(r"scikit-learn|numpy|pandas|scipy|joblib|iterative", line, re.I):
        print(line)
